# Floor plan MVP — обучение YOLO-seg на CubiCasa5K (GPU)

1. **Среда выполнения → Сменить среду выполнения → GPU** (T4 бесплатно; L4/A100 в Colab Pro).
2. Запускайте ячейки по порядку. Веса и логи пишутся на Google Drive в `MyDrive/floorplan/runs`,
   поэтому если сессия оборвётся — просто запустите всё заново: обучение **продолжится** с последней эпохи.
3. Итог: `MyDrive/floorplan/runs/cubicasa_v2/weights/best.pt` — скачать и положить в проект,
   затем `FP_MODEL=<путь>/best.pt uvicorn server.app:app`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Google Drive и зависимости

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE = '/content/drive/MyDrive/floorplan'
os.makedirs(DRIVE, exist_ok=True)
!pip -q install ultralytics shapely lxml

## 2. Код проекта (конвертер CubiCasa → YOLO)
Сгенерировано из `fp/` скриптом `scripts/build_colab.py`.

In [ ]:
import os
os.makedirs('/content/fp', exist_ok=True)
%cd /content

In [ ]:
%%writefile /content/fp/__init__.py
# package

In [ ]:
%%writefile /content/fp/classes.py
"""Список классов MVP и маппинг имён из CubiCasa5K model.svg.

Порядок CLASSES = id класса в YOLO. Не менять порядок после начала обучения.
"""

CLASSES = [
    # конструктив и проёмы
    "wall", "door", "window",
    # сантехника
    "toilet", "sink", "bathtub", "shower",
    # комнаты
    "room_living", "room_bedroom", "room_kitchen", "room_bath",
    "room_entry", "room_storage", "room_outdoor", "room_other",
    # v2 (дообучение поверх v1): новые классы только в конец, чтобы старые id не сдвигались.
    # Скамья сауны: в CubiCasa5K сауну рисуют скамьями (3235 из 5000 планов), а Space Sauna всего 133
    "sauna_bench",
]
CLASS_ID = {name: i for i, name in enumerate(CLASSES)}
ROOM_CLASSES = {c for c in CLASSES if c.startswith("room_")}

# CubiCasa "Space <Name>" -> наш класс комнаты (по мотивам rooms_selected из репозитория CubiCasa5k)
SPACE_MAP = {
    "LivingRoom": "room_living", "Lounge": "room_living", "Dining": "room_living",
    "EatingArea": "room_living",
    "Kitchen": "room_kitchen",
    "Bedroom": "room_bedroom",
    "Bath": "room_bath", "Sauna": "room_bath", "Toilet": "room_bath", "WC": "room_bath",
    "Entry": "room_entry", "HallWay": "room_entry", "Hall": "room_entry", "DraughtLobby": "room_entry",
    "Storage": "room_storage", "Closet": "room_storage", "DressingRoom": "room_storage",
    "Pantry": "room_storage", "Utility": "room_storage",
    "Outdoor": "room_outdoor", "Balcony": "room_outdoor", "Terrace": "room_outdoor",
}
DEFAULT_ROOM = "room_other"
# Space-типы, которые не являются комнатами
SPACE_SKIP = {"Background", "Wall", "Railing", "OpenToBelow", "Below150cm", "Elevated"}

# CubiCasa "FixedFurniture <Name>" -> наш класс (точные имена, сверено с датасетом scripts/inspect_cubicasa.py)
FIXTURE_MAP = {
    "Toilet": "toilet", "Urinal": "toilet",
    "Bathtub": "bathtub", "Jacuzzi": "bathtub",
    "Shower": "shower", "ShowerCab": "shower", "ShowerPlatform": "shower",
    "Sink": "sink", "RoundSink": "sink", "CornerSink": "sink", "DoubleSink": "sink",
    "DoubleSinkRight": "sink", "SideSink": "sink", "WaterTap": "sink",
    "SaunaBench": "sauna_bench", "SaunaBenchLow": "sauna_bench", "SaunaBenchMid": "sauna_bench",
    "SaunaBenchHigh": "sauna_bench",
}
# известные иконки вне MVP — молча пропускаем (ShowerScreen = шторка/стекло, не сам душ)
FIXTURE_SKIP = {"ShowerScreen", "Closet", "CoatCloset", "ElectricalAppliance", "BaseCabinet", "WallCabinet",
                "Housing", "Misc", "Fireplace", "Chimney", "CoatRack", "CounterTop"}

# Синонимы для чата: русские/английские слова -> класс (или группа классов)
SYNONYMS = {
    "wall": ["стен", "wall"],
    "door": ["двер", "door", "проём", "проем"],
    "window": ["окн", "окон", "window"],
    "toilet": ["туалет", "унитаз", "toilet", "wc"],
    "sink": ["раковин", "умывальник", "мойк", "sink"],
    "bathtub": ["ванн", "bathtub", "bath tub"],
    "shower": ["душ", "shower"],
    "room_bedroom": ["спальн", "спален", "bedroom"],
    "room_kitchen": ["кухн", "кухонь", "kitchen"],
    "room_living": ["гостин", "living"],
    "room_bath": ["санузл", "санузел", "ванная комната", "ванные комнаты", "bathroom"],
    "room_entry": ["прихож", "коридор", "холл", "entry", "hall"],
    "room_storage": ["кладов", "гардероб", "storage", "closet"],
    "room_outdoor": ["балкон", "террас", "outdoor"],
    "sauna": ["саун", "sauna", "bastu"],          # виртуальный: помещения со скамьями сауны
    "sauna_bench": ["скамь", "полок"],
    "room": ["комнат", "помещен", "room"],
}

LABELS_RU = {
    "wall": "стены", "door": "двери", "window": "окна", "toilet": "унитазы",
    "sink": "раковины", "bathtub": "ванны", "shower": "душевые",
    "room_living": "гостиные", "room_bedroom": "спальни", "room_kitchen": "кухни",
    "room_bath": "санузлы", "room_entry": "прихожие/коридоры", "room_storage": "кладовые",
    "room_outdoor": "балконы/террасы", "room_other": "прочие помещения", "sauna_bench": "скамьи сауны", "sauna": "сауны",
}

# Цвета оверлея (и для визуальной проверки, и для фронта)
COLORS = {
    "wall": "#333333", "door": "#c0502b", "window": "#3a78b8", "toilet": "#8e44ad",
    "sink": "#16a085", "bathtub": "#2980b9", "shower": "#27ae60",
    "room_living": "#f5d76e", "room_bedroom": "#a3d9a5", "room_kitchen": "#f7b267",
    "room_bath": "#9ad0ec", "room_entry": "#d7bde2", "room_storage": "#cccccc",
    "room_outdoor": "#b5e7a0", "room_other": "#eeeeee", "sauna_bench": "#e59866",
}

In [ ]:
%%writefile /content/fp/cubicasa_svg.py
"""Парсер CubiCasa5K model.svg -> список (класс, полигон) в пикселях картинки.

Структура SVG (по репозиторию CubiCasa5k, floortrans/loaders/house.py):
  <g id="Wall" class="Wall ..."><polygon points="..."/> <g id="Window" ...><polygon/></g> ...</g>
  <g id="Door" class="Door ..."><polygon points="..."/></g>
  <g class="Space Kitchen"><polygon points="..."/></g>
  <g class="FixedFurniture Toilet"><g class="BoundaryPolygon"><polygon/></g></g>

Мы не доверяем этому описанию слепо: scripts/inspect_cubicasa.py считает реальные
имена классов в датасете, а неизвестные имена собираются в `unknown`.
"""
from __future__ import annotations

import re
from collections import Counter
from dataclasses import dataclass, field

from lxml import etree

import numpy as np

from .classes import DEFAULT_ROOM, FIXTURE_MAP, FIXTURE_SKIP, SPACE_MAP, SPACE_SKIP

Polygon = list[tuple[float, float]]


@dataclass
class ParsedSvg:
    objects: list[tuple[str, Polygon]] = field(default_factory=list)
    unknown: Counter = field(default_factory=Counter)  # имена, которые не смаппились
    width: float | None = None   # ВНИМАНИЕ: в CubiCasa это габарит содержимого, а не размер картинки.
    height: float | None = None  # Координаты полигонов = пиксели F1_scaled.png.


_NUM = re.compile(r"-?\d+(?:\.\d+)?(?:e-?\d+)?", re.I)


def parse_points(s: str) -> Polygon:
    nums = [float(x) for x in _NUM.findall(s or "")]
    return [(nums[i], nums[i + 1]) for i in range(0, len(nums) - 1, 2)]


def _local(tag) -> str:
    return tag.split("}")[-1] if isinstance(tag, str) else ""


_TF = re.compile(r"(matrix|translate|scale|rotate)\s*\(([^)]*)\)")


def parse_transform(s: str | None) -> np.ndarray:
    """SVG transform -> 3x3 матрица."""
    M = np.eye(3)
    for kind, args in _TF.findall(s or ""):
        v = [float(x) for x in _NUM.findall(args)]
        T = np.eye(3)
        if kind == "matrix" and len(v) == 6:
            T[0, :] = v[0], v[2], v[4]
            T[1, :] = v[1], v[3], v[5]
        elif kind == "translate" and v:
            T[0, 2], T[1, 2] = v[0], v[1] if len(v) > 1 else 0.0
        elif kind == "scale" and v:
            T[0, 0], T[1, 1] = v[0], v[1] if len(v) > 1 else v[0]
        elif kind == "rotate" and v:
            a = np.radians(v[0])
            R = np.array([[np.cos(a), -np.sin(a), 0], [np.sin(a), np.cos(a), 0], [0, 0, 1]])
            if len(v) == 3:
                R = np.array([[1, 0, v[1]], [0, 1, v[2]], [0, 0, 1]]) @ R @ np.array([[1, 0, -v[1]], [0, 1, -v[2]], [0, 0, 1]])
            T = R
        M = M @ T
    return M


def ctm(el) -> np.ndarray:
    """Полная матрица преобразования элемента (от корня до него включительно)."""
    M = np.eye(3)
    for a in reversed([el, *el.iterancestors()]):
        if a.get("transform"):
            M = M @ parse_transform(a.get("transform"))
    return M


def apply(M: np.ndarray, poly: Polygon) -> Polygon:
    if np.allclose(M, np.eye(3)):
        return poly
    P = np.c_[np.asarray(poly, float), np.ones(len(poly))] @ M.T
    return [(float(x), float(y)) for x, y in P[:, :2]]


def _first_polygon(el) -> Polygon | None:
    """Первый дочерний <polygon> (не рекурсивно в чужие объекты)."""
    for ch in el:
        if _local(ch.tag) == "polygon":
            pts = parse_points(ch.get("points", ""))
            if len(pts) >= 3:
                return pts
    return None


def _boundary_polygon(el) -> Polygon | None:
    """Контур иконки в ЛОКАЛЬНЫХ координатах (transform применяется снаружи).
    BoundaryPolygon/<polygon>; если там rect/path — габарит из <desc> (Width/Depth) или из rect-ов."""
    bp = next((g for g in el.iter() if _local(g.tag) == "g" and "BoundaryPolygon" in (g.get("class") or "")), None)
    p = _first_polygon(bp) if bp is not None else None
    if p:
        return p
    p = _first_polygon(el)
    if p:
        return p
    desc = next((d.text for d in el if _local(d.tag) == "desc" and d.text), "")
    m = dict(re.findall(r"(\w+):\s*(-?[\d.]+)", desc))
    if "Width" in m and "Depth" in m:
        w, d = float(m["Width"]), float(m["Depth"])
        return [(0, 0), (w, 0), (w, d), (0, d)]
    xs, ys = [], []
    for r in (bp if bp is not None else el).iter():
        if _local(r.tag) == "rect":
            x, y = float(r.get("x", 0)), float(r.get("y", 0))
            xs += [x, x + float(r.get("width", 0))]
            ys += [y, y + float(r.get("height", 0))]
    if xs and max(xs) > min(xs) and max(ys) > min(ys):
        return [(min(xs), min(ys)), (max(xs), min(ys)), (max(xs), max(ys)), (min(xs), max(ys))]
    return None


def _float(v):
    try:
        return float(re.sub(r"[^\d.]", "", v)) if v else None
    except ValueError:
        return None


def parse_cubicasa_svg(path_or_bytes) -> ParsedSvg:
    if isinstance(path_or_bytes, (bytes, bytearray)):
        root = etree.fromstring(path_or_bytes)
    else:
        root = etree.parse(str(path_or_bytes)).getroot()

    out = ParsedSvg(width=_float(root.get("width")), height=_float(root.get("height")))
    if (out.width is None or out.height is None) and root.get("viewBox"):
        vb = [float(x) for x in root.get("viewBox").split()]
        out.width, out.height = vb[2], vb[3]

    for el in root.iter():
        if _local(el.tag) != "g":
            continue
        gid = el.get("id") or ""
        cls = (el.get("class") or "").strip()
        first = cls.split(" ")[0] if cls else ""

        if gid == "Wall" or first == "Wall":
            p = _first_polygon(el)
            if p:
                out.objects.append(("wall", apply(ctm(el), p)))
        elif gid == "Window" or first == "Window":
            p = _first_polygon(el)
            if p:
                out.objects.append(("window", apply(ctm(el), p)))
        elif gid == "Door" or first == "Door":
            p = _first_polygon(el)
            if p:
                out.objects.append(("door", apply(ctm(el), p)))
        elif first == "Space":
            name = cls.split(" ")[1] if " " in cls else "Undefined"
            if name in SPACE_SKIP:
                continue
            p = _first_polygon(el)
            if not p:
                continue
            target = SPACE_MAP.get(name)
            if target is None:
                out.unknown[f"Space {name}"] += 1
                target = DEFAULT_ROOM
            out.objects.append((target, apply(ctm(el), p)))
        elif first == "FixedFurniture":
            name = cls.split(" ")[1] if " " in cls else ""
            if name in FIXTURE_SKIP:
                continue
            target = FIXTURE_MAP.get(name)
            if target is None:
                out.unknown[f"FixedFurniture {name}"] += 1
                continue
            p = _boundary_polygon(el)
            if p:
                out.objects.append((target, apply(ctm(el), p)))
            else:
                out.unknown[f"no-shape {name}"] += 1
    return out

In [ ]:
%%writefile /content/fp/convert_yolo.py
"""CubiCasa5K -> датасет Ultralytics YOLO-seg.

    python -m fp.convert_yolo --src data/cubicasa5k --out data/yolo --image F1_scaled.png

Для каждой папки с model.svg пишет labels/<split>/<name>.txt со строками
`class_id x1 y1 x2 y2 ...` (нормировано 0..1) и симлинк на картинку в images/<split>/.
Сплиты берутся из train.txt/val.txt/test.txt датасета, если они есть.
"""
from __future__ import annotations

import argparse
import os
import random
from collections import Counter
from pathlib import Path

import cv2

from .classes import CLASS_ID, CLASSES
from .cubicasa_svg import parse_cubicasa_svg


def find_samples(src: Path, image_name: str):
    for svg in sorted(src.rglob("model.svg")):
        img = svg.parent / image_name
        if not img.exists():
            alt = [p for p in svg.parent.glob("F1_*.png")]
            if not alt:
                continue
            img = alt[0]
        yield svg.parent, svg, img


def read_splits(src: Path) -> dict[str, str]:
    """folder(rel path) -> split, по файлам train.txt / val.txt / test.txt."""
    m = {}
    for split in ("train", "val", "test"):
        f = src / f"{split}.txt"
        if f.exists():
            for line in f.read_text().split():
                m[line.strip("/")] = split
    return m


def sample_name(folder: Path, src: Path) -> str:
    return str(folder.relative_to(src)).strip("/").replace(os.sep, "__")


def svg_to_image_scale(folder: Path, img: Path, w: int, h: int) -> tuple[float, float]:
    """Координаты model.svg = пиксели F1_scaled.png (width/height SVG — лишь габарит содержимого!).
    Для другой картинки (F1_original) — пересчёт через размер F1_scaled."""
    ref = folder / "F1_scaled.png"
    if img.name == ref.name or not ref.exists():
        return 1.0, 1.0
    rh, rw = cv2.imread(str(ref), cv2.IMREAD_UNCHANGED).shape[:2]
    return w / rw, h / rh


def convert_one(svg: Path, img: Path) -> tuple[list[str], Counter, tuple[int, int]]:
    im = cv2.imread(str(img), cv2.IMREAD_UNCHANGED)
    if im is None:
        raise ValueError(f"не читается {img}")
    h, w = im.shape[:2]
    parsed = parse_cubicasa_svg(svg)
    sx, sy = svg_to_image_scale(svg.parent, img, w, h)
    lines = []
    for cls, poly in parsed.objects:
        pts = []
        for x, y in poly:
            pts += [min(max(x * sx / w, 0.0), 1.0), min(max(y * sy / h, 0.0), 1.0)]
        xs, ys = pts[0::2], pts[1::2]
        if max(xs) - min(xs) < 1e-4 or max(ys) - min(ys) < 1e-4:
            continue  # вырожденный полигон
        lines.append(f"{CLASS_ID[cls]} " + " ".join(f"{v:.6f}" for v in pts))
    return lines, parsed.unknown, (w, h)


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--src", required=True, type=Path)
    ap.add_argument("--out", required=True, type=Path)
    ap.add_argument("--image", default="F1_scaled.png")
    ap.add_argument("--limit", type=int, default=0, help="для быстрых проверок")
    ap.add_argument("--val-frac", type=float, default=0.1)
    a = ap.parse_args(argv)

    src, out = a.src.resolve(), a.out.resolve()
    splits = read_splits(src)
    rnd = random.Random(0)
    stats, unknown, n = Counter(), Counter(), 0
    for folder, svg, img in find_samples(src, a.image):
        rel = str(folder.relative_to(src)).strip("/")
        split = splits.get(rel) or ("val" if rnd.random() < a.val_frac else "train")
        name = sample_name(folder, src)
        try:
            lines, unk, _ = convert_one(svg, img)
        except Exception as e:  # битые файлы в датасете бывают
            print(f"[skip] {rel}: {e}")
            continue
        unknown.update(unk)
        (out / "labels" / split).mkdir(parents=True, exist_ok=True)
        (out / "images" / split).mkdir(parents=True, exist_ok=True)
        (out / "labels" / split / f"{name}.txt").write_text("\n".join(lines))
        link = out / "images" / split / f"{name}{img.suffix}"
        if link.exists() or link.is_symlink():
            link.unlink()
        link.symlink_to(img.resolve())
        for ln in lines:
            stats[CLASSES[int(ln.split()[0])]] += 1
        n += 1
        if a.limit and n >= a.limit:
            break

    have = sorted({p.name for p in (out / "images").iterdir()}) if (out / "images").exists() else []
    yaml = [f"path: {out}"]
    yaml += [f"{s}: images/{s}" for s in ("train", "val", "test") if s in have]
    yaml += ["names:"] + [f"  {i}: {c}" for i, c in enumerate(CLASSES)]
    (out / "floorplans.yaml").write_text("\n".join(yaml) + "\n")

    print(f"планов: {n}")
    for c in CLASSES:
        print(f"  {c:14s} {stats[c]}")
    if unknown:
        print("не смаппилось (топ-30):")
        for k, v in unknown.most_common(30):
            print(f"  {k}: {v}")
    print(f"yaml: {out / 'floorplans.yaml'}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/fp/visualize.py
"""Визуальная проверка YOLO-разметки: рисует полигоны поверх картинок.

    python -m fp.visualize --data data/yolo --split train --n 20 --out out/check
"""
from __future__ import annotations

import argparse
from pathlib import Path

import cv2
import numpy as np

from .classes import CLASSES, COLORS


def hex2bgr(h: str):
    h = h.lstrip("#")
    r, g, b = (int(h[i:i + 2], 16) for i in (0, 2, 4))
    return (b, g, r)


def draw_labels(img: np.ndarray, label_text: str) -> np.ndarray:
    h, w = img.shape[:2]
    if img.ndim == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    elif img.shape[2] == 4:
        img = cv2.cvtColor(img, cv2.COLOR_BGRA2BGR)
    fill = img.copy()
    for line in label_text.strip().splitlines():
        v = line.split()
        cls = CLASSES[int(v[0])]
        pts = np.array([[float(v[i]) * w, float(v[i + 1]) * h] for i in range(1, len(v) - 1, 2)], np.int32)
        color = hex2bgr(COLORS[cls])
        cv2.fillPoly(fill, [pts], color)
        cv2.polylines(img, [pts], True, color, 2)
    return cv2.addWeighted(fill, 0.35, img, 0.65, 0)


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True, type=Path)
    ap.add_argument("--split", default="train")
    ap.add_argument("--n", type=int, default=10)
    ap.add_argument("--out", required=True, type=Path)
    a = ap.parse_args(argv)
    a.out.mkdir(parents=True, exist_ok=True)
    imgs = sorted((a.data / "images" / a.split).iterdir())[: a.n]
    for p in imgs:
        lbl = a.data / "labels" / a.split / (p.stem + ".txt")
        img = cv2.imread(str(p))
        vis = draw_labels(img, lbl.read_text() if lbl.exists() else "")
        cv2.imwrite(str(a.out / f"{p.stem}.jpg"), vis)
    print(f"сохранено {len(imgs)} картинок в {a.out}")


if __name__ == "__main__":
    main()

## 3. Датасет CubiCasa5K (5.5 ГБ, Zenodo)
Качается на локальный диск Colab — это быстро. `KEEP_ZIP_ON_DRIVE = True` сохранит архив на Drive, чтобы в следующий раз не качать (нужно ~5.5 ГБ места на Drive).

In [ ]:
KEEP_ZIP_ON_DRIVE = False
URL = 'https://zenodo.org/records/2613548/files/cubicasa5k.zip?download=1'
ZIP_DRIVE = f'{DRIVE}/cubicasa5k.zip'
ZIP = ZIP_DRIVE if os.path.exists(ZIP_DRIVE) else '/content/cubicasa5k.zip'

if not os.path.exists('/content/data/cubicasa5k'):
    if not os.path.exists(ZIP):
        !wget -q --show-progress -c -O "$ZIP" "$URL"
    !mkdir -p /content/data && unzip -q "$ZIP" -d /content/data
    if KEEP_ZIP_ON_DRIVE and ZIP != ZIP_DRIVE:
        !cp "$ZIP" "$ZIP_DRIVE"
!ls /content/data/cubicasa5k && du -sh /content/data/cubicasa5k

## 4. Конвертация в YOLO-seg + визуальная проверка

In [ ]:
%cd /content
# yolo2 = разметка v2 (16 классов, добавлена room_sauna); старая /content/data/yolo — v1, 15 классов
if not os.path.exists('/content/data/yolo2/floorplans.yaml'):
    !python -m fp.convert_yolo --src /content/data/cubicasa5k --out /content/data/yolo2 2>&1 | grep -v libpng
!for s in train val test; do echo $s $(ls /content/data/yolo2/images/$s | wc -l); done

In [ ]:
!python -m fp.visualize --data /content/data/yolo2 --split val --n 4 --out /content/check
from IPython.display import Image, display
import glob
for p in sorted(glob.glob('/content/check/*.jpg'))[:4]:
    display(Image(p, width=700))

## 5. Обучение (v2 = дообучение поверх v1)
- Есть чекпоинт v2 (`runs/cubicasa_v2/weights/last.pt`) — продолжаем его (сессия оборвалась — просто запустите всё заново).
- Иначе есть веса v1 (`runs/cubicasa/weights/best.pt`, 15 классов) — **дообучаем** их на разметке v2:
  Ultralytics переносит голову классов по именам, учится заново только новый класс `room_sauna`.
- Иначе — с нуля от `yolo11s-seg.pt`.
- `batch=-1` — пачка под ~60% памяти GPU; `patience=15` — ранняя остановка.

In [ ]:
from ultralytics import YOLO
RUN = f'{DRIVE}/runs/cubicasa_v2'
last = f'{RUN}/weights/last.pt'
v1 = f'{DRIVE}/runs/cubicasa/weights/best.pt'

if os.path.exists(last):
    print('Продолжаю v2 с', last)
    model = YOLO(last)
    model.train(resume=True)
else:
    start = v1 if os.path.exists(v1) else 'yolo11s-seg.pt'
    print('Старт v2 от', start)
    model = YOLO(start)
    model.train(data='/content/data/yolo2/floorplans.yaml', epochs=60, imgsz=1024, batch=-1,
                device=0, workers=8, patience=15, cache=False, max_det=1000,
                project=f'{DRIVE}/runs', name='cubicasa_v2', exist_ok=True, plots=True)

## 6. Метрики по классам на test (mAP масок)

In [ ]:
best = YOLO(f'{RUN}/weights/best.pt')
m = best.val(data='/content/data/yolo2/floorplans.yaml', split='test', imgsz=1024, device=0, plots=True, max_det=1000,
             project=f'{DRIVE}/runs', name='cubicasa_v2_test', exist_ok=True)
import pandas as pd
rows = [{'class': best.names[c], 'mask_mAP50': m.seg.ap50[i], 'mask_mAP50-95': m.seg.ap[i],
         'box_mAP50': m.box.ap50[i]} for i, c in enumerate(m.ap_class_index)]
pd.DataFrame(rows).round(3)

## 7. Пример предсказания и скачивание весов

In [ ]:
import random
val_imgs = sorted(glob.glob('/content/data/yolo2/images/test/*.png'))
res = best.predict(random.choice(val_imgs), imgsz=1024, conf=0.25, retina_masks=True, max_det=1000)[0]
res.save('/content/pred.jpg')
display(Image('/content/pred.jpg', width=900))

In [ ]:
from google.colab import files
files.download(f'{RUN}/weights/best.pt')